Imports

In [128]:
import pandas as pd
import re
import ast

In [129]:
# load the oscars and movies dataset
oscars = pd.read_csv("the_oscar_award.csv")
df_full = pd.read_csv("movies_dataset.csv")



In [130]:
# aggregate Oscar data to the film-year level.
oscars_film = (
    oscars
    .groupby(["film", "year_film"])
    .agg(
        oscar_nominations=("category", "count"),
        oscar_wins=("winner", "sum")
    )
    .reset_index()
)

In [131]:
oscars_film["oscar_nominated"] = oscars_film["oscar_nominations"] > 0
oscars_film["oscar_won"] = oscars_film["oscar_wins"] > 0

In [132]:
# standardize movie titles for merging.
def clean_title(title):
    title = str(title).lower()
    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"[^a-z0-9 ]", "", title)
    title = title.strip()
    return title

In [133]:
df_full["title_clean"] = df_full["title"].apply(clean_title)
oscars_film["title_clean"] = oscars_film["film"].apply(clean_title)

In [134]:
df_full["release_year"] = pd.to_datetime(
    df_full["release_date"], errors="coerce"
).dt.year

In [135]:
df_merged = df_full.merge(
    oscars_film,
    left_on=["title_clean", "release_year"],
    right_on=["title_clean", "year_film"],
    how="left"
)

In [136]:
# fill missing Oscar information for unmatched movies.
df_merged["oscar_nominations"] = df_merged["oscar_nominations"].fillna(0)
df_merged["oscar_wins"] = df_merged["oscar_wins"].fillna(0)

df_merged["oscar_nominated"] = df_merged["oscar_nominated"].fillna(False)
df_merged["oscar_won"] = df_merged["oscar_won"].fillna(False)

In [137]:
df_merged["oscar_nominated"].mean()

np.float64(0.0413447782546495)

In [ ]:
# Convert stored genre strings into Python lists.
df_merged["genre_list"] = df_merged["genres"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [ ]:
keep_columns = [
    "id",
    "title",
    "release_year",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "genre_list",
    "popularity",
    "oscar_nominations",
    "oscar_wins",
    "oscar_nominated",
    "oscar_won"
]

In [139]:
df_merged = df_merged[keep_columns].copy()

In [141]:
df_merged = df_merged[
    (df_merged["budget"] > 0) 
]

In [142]:
df_merged.to_csv("oscars_movies_merged.csv", index=False)
